# Analitica IPM - Barrio Obrero

Analisis del Indice de Pobreza Multidimensional (IPM) por variable. Primera entrega:
**grafico comparativo Cali vs Barrio Obrero (Comuna 9)** del anio mas reciente.

- Fuente: `data/IPM/Estadisticas_Distritales_IPM_periodica.xlsx`
  (hoja `Anio` = Cali total; hoja `Comuna` = por comuna).
- Incidencia (% de hogares) = Si/total (Cali) o sum/count (Comuna).
- Barrio Obrero corresponde a la **Comuna 9**.

> Nota: la carpeta `data/IPM/` NO se versiona (ver .gitignore). Para Colab se puede subir
> o, si se decide, incluir una copia publica.


## 0. Setup Colab (opcional)

In [ ]:
# === SETUP COLAB (se ignora en local) ===
def _en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
if _en_colab():
    import os, subprocess, sys
    REPO = "https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git"
    DEST = "/content/mapa_Analitico_ciudad_Paraiso"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "matplotlib", "seaborn", "openpyxl"], check=False)
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=False)
    _nb = os.path.join(DEST, "notebooks")
    if os.path.isdir(_nb):
        os.chdir(_nb)
    print("Entorno Colab listo. Directorio:", os.getcwd())
else:
    print("Entorno local (uv). Setup Colab omitido.")


## 1. Entorno

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("Entorno listo.")

## 2. Carga y calculo de incidencias (Cali vs Comuna 9)

In [ ]:
def _ruta_ipm_xlsx(nombre):
    for p in [os.path.join("..", "data", "IPM", nombre),
              os.path.join("data", "IPM", nombre),
              os.path.join("/content", "data", "IPM", nombre),
              os.path.join("/content", "drive", "MyDrive",
                           "mapa_Analitico_ciudad_Paraiso", "data", "IPM", nombre)]:
        if os.path.isfile(p):
            return os.path.abspath(p)
    return None
_rp = _ruta_ipm_xlsx("Estadisticas_Distritales_IPM_periodica.xlsx")
if _rp is None:
    raise FileNotFoundError("No se encontro Estadisticas_Distritales_IPM_periodica.xlsx en data/IPM.")
# Variables del IPM (nombres de columna en el Excel) y etiqueta legible.
VARS = ["Baja_Edu","Analfabetismo","Inasistencia_escolar","Rezago_escolar","CuidadoPriInf",
        "TrabajoInfantil","Desempleo","Trabajo_Informal","Sin_Aseguramiento","Barrera_Acceso_Salud",
        "No_fuente_agua","Inadecuada_eliminacion_excre","Inadecuado_Mat_Pisos",
        "Inadecuado_Mat_Paredes_ext","Hacinamiento_Critico"]
ETIQUETA = {
    "Baja_Edu": "Bajo logro educativo", "Analfabetismo": "Analfabetismo",
    "Inasistencia_escolar": "Inasistencia escolar", "Rezago_escolar": "Rezago escolar",
    "CuidadoPriInf": "Barreras primera infancia", "TrabajoInfantil": "Trabajo infantil",
    "Desempleo": "Desempleo", "Trabajo_Informal": "Trabajo informal",
    "Sin_Aseguramiento": "Sin aseguramiento", "Barrera_Acceso_Salud": "Barreras de salud",
    "No_fuente_agua": "Sin acceso a agua", "Inadecuada_eliminacion_excre": "Eliminacion de excretas",
    "Inadecuado_Mat_Pisos": "Pisos precarios", "Inadecuado_Mat_Paredes_ext": "Paredes precarias",
    "Hacinamiento_Critico": "Hacinamiento critico",
}
# --- CALI (hoja Anio): cada variable tiene columna "Si" (=nombre) y "total" (=siguiente) ---
_a = pd.read_excel(_rp, sheet_name="Anio", header=None)
_hv = _a.iloc[0].tolist()
_col_si = {n: j for j, n in enumerate(_hv) if isinstance(n, str) and n in VARS}
_dat = _a.iloc[3:].copy()
_dat = _dat[pd.to_numeric(_dat[0], errors="coerce").notna()]
_dat[0] = _dat[0].astype(int)
anio_ult = int(_dat[0].max())
_fc = _dat[_dat[0] == anio_ult].iloc[0]
cali = {}
for v in VARS:
    _si = pd.to_numeric(_fc[_col_si[v]], errors="coerce")
    _tot = pd.to_numeric(_fc[_col_si[v] + 1], errors="coerce")
    cali[v] = round(100.0 * _si / _tot, 2) if (pd.notna(_si) and pd.notna(_tot) and _tot) else np.nan
# --- COMUNA 3 (hoja Comuna): columna "sum" (=nombre) y "count" (=siguiente) ---
_c = pd.read_excel(_rp, sheet_name="Comuna", header=None)
_hvc = _c.iloc[0].tolist()
_col_sum = {n: j for j, n in enumerate(_hvc) if isinstance(n, str) and n in VARS}
_dc = _c.iloc[3:].copy()
_dc[0] = _dc[0].ffill()
_dc = _dc[pd.to_numeric(_dc[1], errors="coerce").notna()]
_dc[0] = pd.to_numeric(_dc[0], errors="coerce")
_dc[1] = pd.to_numeric(_dc[1], errors="coerce").astype(int)
_f3 = _dc[(_dc[0] == anio_ult) & (_dc[1] == 9)]
cp = {}
for v in VARS:
    if len(_f3):
        _s = pd.to_numeric(_f3.iloc[0][_col_sum[v]], errors="coerce")
        _n = pd.to_numeric(_f3.iloc[0][_col_sum[v] + 1], errors="coerce")
        cp[v] = round(100.0 * _s / _n, 2) if (pd.notna(_s) and pd.notna(_n) and _n) else np.nan
    else:
        cp[v] = np.nan
df_cmp = pd.DataFrame({"Variable": [ETIQUETA[v] for v in VARS],
                       "Cali": [cali[v] for v in VARS],
                       "Barrio Obrero": [cp[v] for v in VARS]})
print("Anio mas reciente:", anio_ult)
print(df_cmp.to_string(index=False))

## 3. Grafico comparativo - barras agrupadas (Cali vs Barrio Obrero)

In [ ]:
# 9b. Grafico comparativo - barras agrupadas (Cali vs Barrio Obrero)
orden = df_cmp.sort_values('Barrio Obrero', ascending=True)
y = np.arange(len(orden))
h = 0.4

fig, ax = plt.subplots(figsize=(13, 9))
ax.barh(y + h/2, orden['Cali'], height=h, label='Cali (total)',
        color=plt.cm.viridis(0.25), edgecolor='white')
ax.barh(y - h/2, orden['Barrio Obrero'], height=h, label='Barrio Obrero (Comuna 9)',
        color=plt.cm.viridis(0.7), edgecolor='white')
for yi, (vc, vp) in enumerate(zip(orden['Cali'], orden['Barrio Obrero'])):
    if pd.notna(vc): ax.text(vc + 0.6, yi + h/2, f'{vc:.1f}', va='center', fontsize=8)
    if pd.notna(vp): ax.text(vp + 0.6, yi - h/2, f'{vp:.1f}', va='center', fontsize=8, fontweight='bold')
ax.set_yticks(y)
ax.set_yticklabels(orden['Variable'])
ax.set_xlabel('Incidencia (% de hogares)')
ax.set_title(f'Variables IPM: Barrio Obrero vs Cali - {anio_ult}', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
sns.despine()
plt.tight_layout()
plt.show()

## 4. Exportar el grafico (PNG)

In [ ]:
OUT = None
for _b in [os.path.join("..", "data_processed_geo"), "data_processed_geo", "/content/data_processed_geo"]:
    try:
        os.makedirs(_b, exist_ok=True); OUT = os.path.abspath(_b); break
    except Exception:
        continue
_png = os.path.join(OUT, "ipm_comparativo_cali_vs_bo_%d.png" % anio_ult)
fig.savefig(_png, dpi=150, bbox_inches="tight")
print("Grafico exportado:", _png)